# NYC TLC HVFHV monthly 2.5% samples

In Colab, choose **Runtime → Run all**. No file uploads or directory settings are needed after opening this notebook.

- **2025** covers January–December.
- **2026** covers January–September.
- **2024** covers the October–December fallback months.

Completed months are saved as individual CSV files in **Files → hvfhv_samples → monthly**. Rerunning the build reuses verified months and downloads only the ones that still need a sample. To resume across runtime resets, check **persist_months** in the setup cell before building. It uses **My Drive → Colab Notebooks → HVFHV Samples → monthly** and imports completed TLC months from the active runtime.

The notebook creates three ZIPs, each with one combined CSV, in **Files → hvfhv_samples**. It combines each period’s planned months once and checks their row counts before replacing a ZIP. Use the download buttons after the build, or the optional **Save to Google Drive** cell for the ZIPs.

Each month's sample is drawn uniformly without replacement from all rows in its official Parquet file. The sample has `floor(monthly rows / 40)` rows, uses base seed 42 plus a month-specific offset, and retains every original column. The added columns identify the source month, the zero-based source row, and the sampling weight `monthly rows / sampled rows`.

The runtime reads the monthly Parquet files from the official NYC TLC site, one month at a time. Remote reads keep the raw data off disk. If TLC returns HTTP 403, the build pauses at that month. For other range read failures, the notebook tries a temporary download of that month's official TLC file. Rerun the build cell to resume using the selected monthly folder. The 2.5% fraction controls the output row count. The source scan can still transfer most of each monthly file because the sample spans the full month and includes every column.

[NYC TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

In [ ]:
# @title Set up the sampler
import importlib.util
import subprocess
import sys

packages = {
    "numpy": "numpy",
    "pyarrow": "pyarrow",
    "fsspec": "fsspec[http]",
    "aiohttp": "aiohttp",
    "ipywidgets": "ipywidgets",
}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import json
import random
import shutil
import tempfile
import time
import zipfile
from pathlib import Path
from urllib.request import urlopen

import aiohttp
import fsspec
import numpy as np
import pyarrow as pa
import pyarrow.csv as arrow_csv
import pyarrow.parquet as parquet

OUTPUT = Path("/content/hvfhv_samples")
STATE = Path("/content/.hvfhv_samples_work")
LOCAL_MONTHLY = OUTPUT / "monthly"

persist_months = False  # @param {type:"boolean"}
MONTHLY = LOCAL_MONTHLY
if persist_months:
    from google.colab import drive

    mount = Path("/content/drive")
    drive.mount(str(mount))
    MONTHLY = mount / "MyDrive" / "Colab Notebooks" / "HVFHV Samples" / "monthly"

SOURCE = "https://d37ci6vzurychx.cloudfront.net/trip-data"
BASE_SEED = 42
SAMPLE_DENOMINATOR = 40
GROUPS = {
    "hvfhv_2025_2p5pct": [f"2025-{month:02d}" for month in range(1, 13)],
    "hvfhv_2026_jan-sep_2p5pct": [f"2026-{month:02d}" for month in range(1, 10)],
    "hvfhv_2024_oct-dec_2p5pct": [f"2024-{month:02d}" for month in range(10, 13)],
}
EXTRA_FIELDS = [
    ("_sample_source_month", pa.string()),
    ("_sample_source_row", pa.int64()),
    ("_sample_weight", pa.float64()),
]
PARSE_OPTIONS = arrow_csv.ParseOptions(newlines_in_values=True)


In [ ]:
# @title Define sampling, CSV export, and validation
from contextlib import contextmanager
from hashlib import file_digest


@contextmanager
def temporary_output(path):
    with tempfile.NamedTemporaryFile(
        prefix=f".{path.name}.", suffix=".part", dir=path.parent, delete=False
    ) as file:
        pending = Path(file.name)
    try:
        yield pending
    finally:
        pending.unlink(missing_ok=True)


def load_json(path):
    try:
        record = json.loads(path.read_text(encoding="utf-8"))
    except (FileNotFoundError, ValueError):
        return None
    return record if isinstance(record, dict) else None


def write_json(path, record):
    with temporary_output(path) as pending:
        pending.write_text(json.dumps(record), encoding="utf-8")
        pending.replace(path)


def file_sha256(path):
    with path.open("rb") as stream:
        return file_digest(stream, "sha256").hexdigest()


def sample_parquet(source, label, target):
    year, month = map(int, label.split("-"))
    with parquet.ParquetFile(source, pre_buffer=True) as file:
        rows = file.metadata.num_rows
        count = rows // SAMPLE_DENOMINATOR
        if count == 0:
            raise ValueError(f"{label}: {rows:,} source rows yield an empty sample")

        indices = np.asarray(
            sorted(random.Random(BASE_SEED + 100 * year + month).sample(range(rows), count)),
            dtype=np.int64,
        )
        raw_schema = file.schema_arrow
        if len(set(raw_schema.names)) != len(raw_schema.names):
            raise ValueError(f"{label}: duplicate source column names")
        if set(raw_schema.names).intersection(name for name, _ in EXTRA_FIELDS):
            raise ValueError(f"{label}: source contains a reserved sampling metadata column")
        schema = pa.schema([*raw_schema, *(pa.field(name, dtype) for name, dtype in EXTRA_FIELDS)])

        offset = selected = 0
        weight = rows / count
        with arrow_csv.CSVWriter(str(target), schema) as writer:
            for group in range(file.metadata.num_row_groups):
                for batch in file.iter_batches(row_groups=[group], batch_size=131_072):
                    stop = offset + batch.num_rows
                    end = int(np.searchsorted(indices, stop))
                    if end > selected:
                        source_rows = indices[selected:end]
                        picked = batch.take(pa.array(source_rows - offset))
                        arrays = [
                            *picked.columns,
                            pa.array([label] * picked.num_rows, type=pa.string()),
                            pa.array(source_rows, type=pa.int64()),
                            pa.array(np.full(picked.num_rows, weight), type=pa.float64()),
                        ]
                        writer.write_batch(pa.RecordBatch.from_arrays(arrays, schema=schema))
                    selected, offset = end, stop
                    if selected == count:
                        break
                if selected == count:
                    break
        if selected != count:
            raise ValueError(f"{label}: wrote {selected:,} sample rows, expected {count:,}")
    return {
        "month": label,
        "source_rows": rows,
        "sample_rows": count,
        "sample_weight": weight,
        "columns": schema.names,
        "seed": BASE_SEED,
        "sample_denominator": SAMPLE_DENOMINATOR,
    }


from http.client import HTTPException
from urllib.error import HTTPError
from urllib.request import Request

DOWNLOAD_ERRORS = (OSError, HTTPException)
READ_ERRORS = (*DOWNLOAD_ERRORS, ValueError, pa.ArrowException, aiohttp.ClientError)
RANGE_READS_ALLOWED = True


def http_status(error):
    while error is not None:
        status = getattr(error, "status", None) or getattr(error, "code", None)
        if isinstance(status, int):
            return status
        error = error.__cause__ or error.__context__
    return None


def download_source(url, target):
    request = Request(url, headers={"Accept-Encoding": "identity"})
    with urlopen(request, timeout=90) as source, target.open("wb") as dest:
        if source.status != 200:
            raise HTTPError(url, source.status, "Expected a complete file", source.headers, None)
        length = source.headers.get("Content-Length")
        for block in iter(lambda: source.read(4 * 2**20), b""):
            dest.write(block)

    size = target.stat().st_size
    if length is not None and size != int(length):
        raise OSError(f"Incomplete download: {size:,} of {int(length):,} bytes")


def sample_download(url, label, pending):
    with tempfile.TemporaryDirectory(dir=STATE) as folder:
        raw = Path(folder) / "source.parquet"
        for attempt in range(3):
            try:
                download_source(url, raw)
                break
            except DOWNLOAD_ERRORS as error:
                if attempt == 2 or http_status(error) in (401, 403, 404, 410):
                    raise
                time.sleep(2 ** (attempt + 1))
        return sample_parquet(raw, label, pending)


def read_hvfhv_month(url, label, pending):
    global RANGE_READS_ALLOWED

    try:
        if RANGE_READS_ALLOWED:
            try:
                with fsspec.open(
                    url, mode="rb", block_size=8 * 2**20, cache_type="blockcache",
                    cache_options={"maxblocks": 8},
                    client_kwargs={"trust_env": True, "timeout": aiohttp.ClientTimeout(total=90, connect=15)},
                ) as source:
                    return sample_parquet(source, label, pending)
            except READ_ERRORS as error:
                status = http_status(error)
                if status == 403:
                    raise
                if status == 429:
                    RANGE_READS_ALLOWED = False
                pending.unlink(missing_ok=True)
                print(f"{label}: range read failed. Trying one temporary TLC file.", flush=True)

        return sample_download(url, label, pending)
    except READ_ERRORS as error:
        if http_status(error) == 403:
            raise RuntimeError(
                f"{label}: official TLC returned HTTP 403. The build paused at this month. "
                "Rerun the build cell to resume using the selected monthly folder."
            ) from None
        raise RuntimeError(
            f"{label}: official TLC download failed ({error}). "
            "Rerun the build cell to resume using the selected monthly folder."
        ) from None


def completed_month(folder, label, url):
    monthly = folder / f"{label}.csv"
    record = load_json(folder / f"{label}.json")
    if record is None or not monthly.is_file():
        return None

    try:
        if (
            record["month"] != label
            or record["url"] != url
            or record.get("retrieval_url", url) != url
            or record["seed"] != BASE_SEED
            or record["sample_denominator"] != SAMPLE_DENOMINATOR
            or record["sample_rows"] <= 0
            or record["sample_rows"] != record["source_rows"] // SAMPLE_DENOMINATOR
            or record["csv_bytes"] != monthly.stat().st_size
            or not isinstance(record["columns"], list)
        ):
            return None
        if "csv_sha256" in record and record["csv_sha256"] != file_sha256(monthly):
            return None
    except (FileNotFoundError, TypeError, KeyError):
        return None
    return record


def publish_month(monthly, pending, record):
    size = pending.stat().st_size
    digest = file_sha256(pending)
    if size != record.get("csv_bytes", size) or digest != record.get("csv_sha256", digest):
        raise OSError(f"{monthly.name}: incomplete or inconsistent checkpoint copy")
    record = {**record, "csv_bytes": size, "csv_sha256": digest}
    pending.replace(monthly)
    write_json(monthly.with_suffix(".json"), record)
    return record


def sample_month(label):
    url = f"{SOURCE}/fhvhv_tripdata_{label}.parquet"
    monthly = MONTHLY / f"{label}.csv"
    for folder in dict.fromkeys((MONTHLY, LOCAL_MONTHLY, STATE)):
        record = completed_month(folder, label, url)
        if record is None:
            continue
        if folder != MONTHLY:
            with temporary_output(monthly) as pending:
                shutil.copyfile(folder / monthly.name, pending)
                record = publish_month(monthly, pending, record)
        print(f"{label}: reusing {record['sample_rows']:,} completed sample rows", flush=True)
        return record

    print(f"{label}: reading the official TLC Parquet", flush=True)
    with temporary_output(monthly) as pending:
        record = read_hvfhv_month(url, label, pending)
        record["url"] = url
        record = publish_month(monthly, pending, record)

    print(f"{label}: {record['sample_rows']:,} sampled out of {record['source_rows']:,} rows", flush=True)
    return record


def write_combined_csv(sink, records):
    columns = records[0]["columns"]
    if all(record["columns"] == columns for record in records):
        for i, record in enumerate(records):
            with (MONTHLY / f"{record['month']}.csv").open("rb") as source:
                if i:
                    source.readline()
                shutil.copyfileobj(source, sink, length=4 * 2**20)
        return

    # Keep the union of the original columns if a year's monthly schemas change.
    columns = list(dict.fromkeys(name for record in records for name in record["columns"]))
    schema = pa.schema([(name, pa.string()) for name in columns])
    with arrow_csv.CSVWriter(sink, schema) as writer:
        for record in records:
            options = arrow_csv.ConvertOptions(
                column_types={name: pa.string() for name in record["columns"]},
                strings_can_be_null=True, quoted_strings_can_be_null=False, null_values=[""],
            )
            with arrow_csv.open_csv(
                MONTHLY / f"{record['month']}.csv", parse_options=PARSE_OPTIONS, convert_options=options
            ) as reader:
                for batch in reader:
                    fields = dict(zip(batch.schema.names, batch.columns))
                    arrays = [
                        fields[name] if name in fields else pa.nulls(batch.num_rows, type=pa.string())
                        for name in columns
                    ]
                    writer.write_batch(pa.RecordBatch.from_arrays(arrays, schema=schema))


def pack_group(stem, records):
    if [record["month"] for record in records] != GROUPS[stem]:
        raise ValueError(f"{stem}: expected one checkpoint for each planned month in order")

    path = OUTPUT / f"{stem}.zip"
    report = STATE / f"{stem}.zip.json"
    saved = load_json(report)
    if (
        path.is_file()
        and saved
        and saved.get("records") == records
        and saved.get("bytes") == path.stat().st_size
        and saved.get("sha256") == file_sha256(path)
    ):
        return path

    member = f"{stem}.csv"
    with temporary_output(path) as pending:
        print(f"{stem}: compressing and checking the CSV", flush=True)
        level = 3 if stem.startswith("hvfhv_2026_") else 6
        with zipfile.ZipFile(pending, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=level) as archive:
            with archive.open(member, "w", force_zip64=True) as sink:
                write_combined_csv(sink, records)

        expected = {r["month"]: r["sample_rows"] for r in records}
        found = dict.fromkeys(expected, 0)
        name = "_sample_source_month"
        options = arrow_csv.ConvertOptions(include_columns=[name], column_types={name: pa.string()})
        with zipfile.ZipFile(pending) as archive, archive.open(member) as stream:
            with arrow_csv.open_csv(stream, parse_options=PARSE_OPTIONS, convert_options=options) as reader:
                for batch in reader:
                    for item in batch.column(0).value_counts().to_pylist():
                        found[item["values"]] += item["counts"]
        if expected != found:
            raise ValueError(f"{stem}: the exported CSV does not match the expected monthly row counts")

        complete = {"bytes": pending.stat().st_size, "sha256": file_sha256(pending), "records": records}
        pending.replace(path)
        write_json(report, complete)
    return path


In [ ]:
# @title Build and verify the three CSV ZIPs
RANGE_READS_ALLOWED = False
OUTPUT.mkdir(parents=True, exist_ok=True)
STATE.mkdir(parents=True, exist_ok=True)
MONTHLY.mkdir(parents=True, exist_ok=True)
print(f"Monthly checkpoints: {MONTHLY}", flush=True)

OUTPUT_FILES = []
for stem, labels in GROUPS.items():
    rows = [sample_month(label) for label in labels]
    archive = pack_group(stem, rows)
    OUTPUT_FILES.append(archive)
    print(f"Ready: {archive.name} | {sum(r['sample_rows'] for r in rows):,} rows | "
          f"{archive.stat().st_size / 1e6:.1f} MB", flush=True)

print(f"All three ZIPs are in the Colab Files panel: {OUTPUT}")


Monthly checkpoints: /content/hvfhv_samples/monthly
2025-01: reusing 510,141 completed sample rows
2025-02: reusing 483,486 completed sample rows
2025-03: reusing 513,421 completed sample rows
2025-04: reusing 493,849 completed sample rows
2025-05: reusing 527,279 completed sample rows
2025-06: reading the official TLC Parquet
2025-06: 496,700 sampled out of 19,868,009 rows
2025-07: reading the official TLC Parquet
2025-07: 491,325 sampled out of 19,653,012 rows
2025-08: reading the official TLC Parquet
2025-08: 481,786 sampled out of 19,271,461 rows
2025-09: reading the official TLC Parquet
2025-09: 485,866 sampled out of 19,434,641 rows
2025-10: reading the official TLC Parquet
2025-10: 532,717 sampled out of 21,308,701 rows
2025-11: reading the official TLC Parquet
2025-11: 520,456 sampled out of 20,818,240 rows
2025-12: reading the official TLC Parquet
2025-12: 552,710 sampled out of 22,108,438 rows
hvfhv_2025_2p5pct: compressing and checking the CSV
Ready: hvfhv_2025_2p5pct.zip | 

In [10]:
# @title Download the ZIPs
from google.colab import files
from IPython.display import display
import ipywidgets as widgets

buttons = []
download_status = widgets.Output()

def download_click(button):
    with download_status:
        download_status.clear_output(wait=True)
        path = OUTPUT / button.description
        print(f"Starting download: {path.name}")
        files.download(str(path))

for path in OUTPUT_FILES:
    button = widgets.Button(description=path.name, layout=widgets.Layout(width="auto"))
    button.on_click(download_click)
    buttons.append(button)

print("Click a filename to download it. The same ZIPs appear under Files → hvfhv_samples.")
display(widgets.VBox([*buttons, download_status]))


Click a filename to download it. The same ZIPs appear under Files → hvfhv_samples.


In [ ]:
# @title Save a permanent copy to Google Drive (optional)
save_to_drive = False  # @param {type:"boolean"}

if save_to_drive:
    from google.colab import drive

    mount = Path("/content/drive")
    drive.mount(str(mount))
    destination = mount / "MyDrive" / "Colab Notebooks" / "HVFHV Samples"
    destination.mkdir(parents=True, exist_ok=True)
    for path in OUTPUT_FILES:
        temporary = destination / f"{path.name}.part"
        try:
            shutil.copyfile(path, temporary)
            if temporary.stat().st_size != path.stat().st_size:
                raise OSError(f"Drive copy was incomplete: {path.name}")
            temporary.replace(destination / path.name)
        finally:
            temporary.unlink(missing_ok=True)
    print(f"Saved all three ZIPs to {destination}")
else:
    print("For a copy that survives runtime reset, check save_to_drive and run this cell.")
